# The optimization workflow

**Goal.** Run an optimization with APEX from start to finish: copy an example, read its configuration, validate it, submit it, wait for it and read the result. You will run two optimizers on Rosenbrock's function, edit copies of the configuration and rerun them one option at a time, then run two global-search optimizers on Rastrigin's function, where the shape of the problem matters. At the end the same runs are looked at from the terminal.

**Prerequisites.** APEX installed and basic Python. The examples use analytic test functions, so there is no simulator to set up.

**Words used here.**

- The *controls* are the variables the optimizer changes to lower the objective. In the configuration they are declared in `[[controls.variables]]`.
- The *starting point* of an optimization is the `initial` values of the controls.
- *PET* is the toolbox whose optimizers (EnOpt, GenOpt, SmcOpt and others) APEX runs for you. You choose one by name in the configuration.
- *EnOpt* is an *ensemble* method: at each iteration it draws `ne` random samples of the controls around the current point, evaluates the objective on each, and estimates the gradient from the results.
- A *trust region* method is deterministic: it works with a local model of the objective and a radius within which it trusts that model.
- *Backtracking* means that when a trial step does not improve the objective enough, the optimizer tries a shorter step.
- A *local minimum* is a point lower than all its neighbours but not necessarily the lowest point overall (the *global minimum*). A problem with many local minima is *multimodal*.
- The *projected gradient* is the gradient restricted to directions that keep the controls between `lower` and `upper`. When it is near 0, no allowed step improves the objective.

**Where files go.** The notebook uses APEX's Python client on a *services root* of its own: a folder where APEX keeps its jobs and results. Here it is `apex_home`, and example copies go in `work`. Both are relative to the notebook's working folder.

**Judge the answer by the objective.** The status of a job and the state of a run (`finished` means the optimizer ended on its own rules) tell you about the work, not about the answer. Only the objective value says how close you got to the minimum.

In [1]:
from pathlib import Path


def rel(p):
    """Show a path relative to the working folder (it may already be relative)."""
    p = Path(p)
    try:
        return p.relative_to(Path.cwd())
    except ValueError:
        return p


print("helper ready")

helper ready


## Step 1: copy an example from the catalogue

APEX ships a catalogue of examples. Copying one into a working folder gives you configuration files you can read and edit.

The example `optimization` holds two configurations of the same problem, Rosenbrock's function minimised by EnOpt and by a trust region: `rosenbrock_enopt.toml` and `rosenbrock_trust_region.toml`.

In [2]:
import math

import pandas as pd

from apex.client import Client
from apex.services.examples import copy_example, examples_root, find_examples
from apex.services.settings import ExamplesSection

catalogue = {e.id: e for e in find_examples(examples_root(ExamplesSection()))}
copied = copy_example(catalogue["optimization"], Path("work"))
config = copied.folder / "rosenbrock_enopt.toml"
config_trust = copied.folder / "rosenbrock_trust_region.toml"
print(rel(config))
print(rel(config_trust))

work/optimization/rosenbrock_enopt.toml
work/optimization/rosenbrock_trust_region.toml


## Step 2: read the configuration

An optimization is one TOML file: the objective, the controls and the optimizer with its options. Print the EnOpt file without its comments to see the settings alone.

In [3]:
def setting(line):
    for index, char in enumerate(line):
        if char == "#" and line[:index].count('"') % 2 == 0:
            return line[:index].rstrip()
    return line.rstrip()


settings = [setting(line) for line in config.read_text().splitlines()]
print("\n".join(line for line in settings if line))

[optimization]
name = "rosenbrock_enopt"
seed = 2026
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
lower = -2.0
upper = 2.0
variance = 0.001
[objective]
name = "rosenbrock"
[optimizer]
name = "enopt"
options = { ne = 20, maxiter = 300, alpha = 0.05, optimizer = "Adam", beta = 0.9, resample = 2 }


How to read it:

- `[optimization]` gives the run a `name` and a `seed`, so a run can be repeated.
- `[[controls.variables]]` declares a variable `x`. Its `initial` list has four values, so the problem has four controls, which the results name `x[0]` to `x[3]`. `lower` and `upper` bound them. `variance` sets how widely EnOpt's samples spread around the current point.
- `[objective]` names the function to minimise, here `rosenbrock`.
- `[optimizer]` names the method and passes `options`. For `enopt`:
  - `ne` is the ensemble size (default 100) and `maxiter` the iteration limit (default 100).
  - `alpha` is the initial step size (default 0.1), `optimizer` the step rule (`GD`, `Adam`, ...) and `beta` the momentum.
  - Backtracking: each attempt estimates a gradient and tries a step; if the trial does not improve the mean objective enough, it tries a shorter one, up to `alpha_maxiter` trials (default 5). `resample` (default 0) is how many times a new ensemble is then drawn for another attempt, with the sample covariance shrunk by `cov_factor` (default 0.5) each time.
  - `tol` (unset by default) is the improvement of the mean objective a trial must achieve to be accepted.
  - `ftol`, `xtol` and `gtol` are convergence tolerances: when the relative change in the objective, the change in the controls or the projected gradient falls below them, the run stops before `maxiter`.

With four controls, Rosenbrock's minimum is at `x = (1, 1, 1, 1)`, where the objective is 0. At the starting point `(-1.2, 1.0, -0.5, 0.8)` the objective is about 282. The function has a single minimum, but it lies at the end of a long, curved, nearly flat valley. Methods tend to stop somewhere along the valley rather than at its end.

Now the trust-region file. The objective and controls are the same; the optimizer section differs. Not every optimizer takes the same options, so check the reference for the one you choose.

In [4]:
settings_trust = [setting(line) for line in config_trust.read_text().splitlines()]
print("\n".join(line for line in settings_trust if line))

[optimization]
name = "rosenbrock_trust_region"
seed = 1
[[controls.variables]]
name = "x"
initial = [-1.2, 1.0, -0.5, 0.8]
[objective]
name = "rosenbrock"
[optimizer]
name = "trust_region"
options = { maxiter = 100 }


## Step 3: validate before running

Validation checks the configuration and reports a plan, without simulating anything. It is cheap, so make it a habit: a typo in a name shows up here at once instead of after a long run.

The plan has a `max_evaluations` field, the most objective evaluations the plan allows. It can be `None`, which means the plan states no figure; the helper below says so.

In [5]:
client = Client(Path("apex_home"))  # a services root of its own


def describe(plan):
    budget = plan.max_evaluations
    shown = "no figure in the plan" if budget is None else f"max_evaluations = {budget}"
    print(f"{plan.optimizer} on {plan.objective}: {plan.n_controls} controls, {shown}")


checked = client.validate_optimization(config)
print("valid:", checked.valid)
plan = checked.optimization
describe(plan)

valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 23402


## Step 4: submit and wait

`client.submit_optimization` takes the configuration and returns a handle that carries a job id. `client.wait` waits on that job, up to the timeout, and returns its status.

- If the wait ended while the job was still going, call `client.wait(handle.id, ...)` again with a longer timeout.
- If the job did not go through, the cell prints the whole status, which carries the details, after looking for a run with the job's id among `client.optimizations()`.

In [6]:
handle = client.submit_optimization(config)
status = client.wait(handle.id, timeout_s=600, poll_s=0.5)
print("job state:", status.state)
rows = [r for r in client.optimizations() if r["job"] == handle.id]
if rows:
    print("run folder:", rel(rows[0]["path"]))
else:
    print("no run folder was created for this job")
    print(status)

job state: complete
run folder: apex_home/optimizations/rosenbrock_enopt/204b77f691e3


## Step 5: read the result

Find the run of our job by its job id among `client.optimizations()`, and ask for its summary. It holds the run's state, the final objective value and the controls found.

In [7]:
row = next(r for r in client.optimizations() if r["job"] == handle.id)
summary = client.optimization_summary(Path(row["path"]))
print("run state:", summary.state, "|", summary.optimizer, "on", summary.objective)
print("objective:", summary.result["objective"])
names, values = summary.result["names"], summary.result["x"]
print("controls:", dict(zip(names, values, strict=True)))
print("run folder:", rel(row["path"]))

run state: finished | enopt on rosenbrock
objective: 0.9878013710933521
controls: {'x[0]': 0.6878032981993977, 'x[1]': 0.48472813220308203, 'x[2]': 0.26026533786111106, 'x[3]': 0.0680490417549587}
run folder: apex_home/optimizations/rosenbrock_enopt/204b77f691e3


Compare with the known answer: the minimum is 0 at `(1, 1, 1, 1)`.

How to judge the number, for this problem:

- The objective started near 282, so a final value below 1 is a fall of more than 99%. That is real progress, but it is not the minimum: the controls printed above are still far from 1, and a value around 0.99 is clearly short of 0.
- As a rough rule for Rosenbrock with four controls, a value below about 1e-3 with every control close to 1 is near the minimum. A value of 0.1 or more means the run stopped somewhere along the valley, and it is worth trying the edits in step 6.
- Stopping along the valley is common for Rosenbrock and is not a sign that something went wrong.

A run submitted through the client lives in `apex_home/optimizations/<name>/<id>`; the id differs from run to run.

## Step 6: edit a copy and rerun

To try to get closer, first give EnOpt a larger iteration limit. Keep every run separate: write a copy of the file with a new `name` and a larger `maxiter`, then validate and submit the copy exactly as before. A run already has its own directory, so do not reuse a name.

The helper `run_copy` bundles validate, submit, wait and summary. The cell after it writes the copies and compares the objectives. It also writes a second copy, `rosenbrock_enopt_cli.toml`, which it does not run; the terminal section uses it.

In [8]:
def run_copy(cfg):
    """Validate, submit and wait for one configuration; return its plan and summary."""
    check = client.validate_optimization(cfg)
    print("valid:", check.valid)
    describe(check.optimization)
    job = client.submit_optimization(cfg)
    final = client.wait(job.id, timeout_s=900, poll_s=0.5)
    print("job state:", final.state)
    found = next(r for r in client.optimizations() if r["job"] == job.id)
    return check.optimization, client.optimization_summary(Path(found["path"]))


text = config.read_text()
long_text = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_long"')
long_text = long_text.replace("maxiter = 300", "maxiter = 1000")
config_long = copied.folder / "rosenbrock_enopt_long.toml"
config_long.write_text(long_text)

# A second copy for the terminal section below (own name; not run here).
cli_text = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_cli"')
config_cli = copied.folder / "rosenbrock_enopt_cli.toml"
config_cli.write_text(cli_text)

print("changed:", long_text != text)
plan_long, summary_long = run_copy(config_long)
before = summary.result["objective"]
after = summary_long.result["objective"]
print("run state:", summary_long.state, "objective:", after)
if after < before:
    print(f"maxiter 1000 improved the objective: {before:.6f} -> {after:.6f}")
elif after == before:
    print("identical objective: the larger limit changed nothing")
else:
    print(f"the objective did not improve: {before:.6f} -> {after:.6f}")

changed: True
valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 78002


job state: complete
run state: finished objective: 0.9878013710933521
identical objective: the larger limit changed nothing


Read the last line. If the objective improved, the extra iterations were used. If it is identical to step 5, the larger `maxiter` made no difference: the first run ended before reaching its limit, because one of the optimizer's own stopping rules (`ftol`, `xtol` or `gtol`) fired. Then a bigger iteration limit is not the way forward; change the step size or the seed instead.

Here is one concrete edit: a smaller initial step `alpha` (0.02 instead of 0.05) and another `seed` (7), in a renamed copy. A smaller step is more cautious along the curved valley; a different seed draws different ensembles.

In [9]:
alt_text = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_alpha002"')
alt_text = alt_text.replace("alpha = 0.05", "alpha = 0.02").replace("seed = 2026", "seed = 7")
config_alt = copied.folder / "rosenbrock_enopt_alpha002.toml"
config_alt.write_text(alt_text)
print("changed:", alt_text != text)

plan_alt, summary_alt = run_copy(config_alt)
print("objective, alpha 0.05, seed 2026:", summary.result["objective"])
print("objective, alpha 0.02, seed 7:   ", summary_alt.result["objective"])

changed: True
valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 23402


job state: complete
objective, alpha 0.05, seed 2026: 0.9878013710933521
objective, alpha 0.02, seed 7:    3.7018071130500143


### Reading a worse result: change one option at a time

If the changed run ends with a higher objective than the original, the combination did not help: it stopped further from the minimum 0. That does not say which change caused it. A smaller step makes slower progress along the valley, and a different seed alone can move the result, because EnOpt is random. To find out, rerun with each change alone, each in its own renamed copy:

- the smaller `alpha` with the original seed 2026;
- the original `alpha` with seed 7.

The cell below runs both and prints all four objectives side by side.

In [10]:
only_alpha = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_only_alpha"')
only_alpha = only_alpha.replace("alpha = 0.05", "alpha = 0.02")
config_only_alpha = copied.folder / "rosenbrock_enopt_only_alpha.toml"
config_only_alpha.write_text(only_alpha)

only_seed = text.replace('name = "rosenbrock_enopt"', 'name = "rosenbrock_enopt_only_seed"')
only_seed = only_seed.replace("seed = 2026", "seed = 7")
config_only_seed = copied.folder / "rosenbrock_enopt_only_seed.toml"
config_only_seed.write_text(only_seed)

plan_oa, summary_oa = run_copy(config_only_alpha)
plan_os, summary_os = run_copy(config_only_seed)

base = summary.result["objective"]
obj_oa = summary_oa.result["objective"]
obj_os = summary_os.result["objective"]
single = pd.DataFrame(
    [
        {"alpha": 0.05, "seed": 2026, "objective": base},
        {"alpha": 0.02, "seed": 2026, "objective": obj_oa},
        {"alpha": 0.05, "seed": 7, "objective": obj_os},
        {"alpha": 0.02, "seed": 7, "objective": summary_alt.result["objective"]},
    ]
)
print(single.to_string(index=False))

if obj_oa > base and obj_os <= base:
    print("the smaller alpha is what made the run worse")
elif obj_os > base and obj_oa <= base:
    print("the seed is what made the run worse: the original may have been a lucky draw")
elif obj_os > base and obj_oa > base:
    print("each change alone ends worse than the original run")
else:
    print("neither change alone is worse than the original; only the combination hurt")

valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 23402


job state: complete
valid: True
enopt on rosenbrock: 4 controls, max_evaluations = 23402


job state: complete
 alpha  seed  objective
  0.05  2026   0.987801
  0.02  2026   3.684919
  0.05     7   0.617848
  0.02     7   3.701807
the smaller alpha is what made the run worse


The last line tells you which change to blame. If the seed alone moves the objective a lot, one run of EnOpt says little: repeat it with several seeds before drawing conclusions.

## Step 7: run the other optimizer and compare

The same steps work for the trust-region configuration.

In [11]:
plan_trust, summary_trust = run_copy(config_trust)

labelled = (
    ("maxiter 300", plan, summary),
    ("maxiter 1000", plan_long, summary_long),
    ("alpha 0.02, seed 7", plan_alt, summary_alt),
    ("maxiter 100", plan_trust, summary_trust),
)
compare = pd.DataFrame(
    [
        {
            "optimizer": s.optimizer,
            "run": label,
            "planned max evaluations": p.max_evaluations,
            "state": s.state,
            "objective": s.result["objective"],
            "controls": [round(float(v), 3) for v in s.result["x"]],
        }
        for label, p, s in labelled
    ]
)
print(compare.to_string(index=False))

valid: True
trust_region on rosenbrock: 4 controls, max_evaluations = 502


job state: complete
   optimizer                run  planned max evaluations    state  objective                      controls
       enopt        maxiter 300                    23402 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
       enopt       maxiter 1000                    78002 finished   0.987801   [0.688, 0.485, 0.26, 0.068]
       enopt alpha 0.02, seed 7                    23402 finished   3.701807   [-0.573, 0.34, 0.13, 0.015]
trust_region        maxiter 100                      502 finished   3.701429 [-0.776, 0.613, 0.382, 0.146]


Reading the table:

- Compare the `objective` column: a lower value is closer to the minimum 0, under each run's own stopping rules.
- `planned max evaluations` is a ceiling, not what was used. EnOpt evaluates an ensemble at every iteration, so its ceiling is much larger than the trust-region plan's. A run that stops on a tolerance uses less than its ceiling.
- The conclusion is modest: the table shows how far each configuration got, not which method is better in general. For a fair comparison, give the methods the same budget measured the same way and repeat with several seeds.

### Trying a larger `trust_radius`

If the trust-region objective is clearly above 0, it may have stopped early. `trust_region` takes these options (all go in the `options = { ... }` line of `[optimizer]`):

- `trust_radius`: the initial radius of the region where the local model is trusted (default 1.0). `trust_radius_max` and `trust_radius_min` limit it (defaults 100 and 1/1000 times the initial radius); the run stops when the radius reaches the minimum.
- `ftol` (default 1e-5), `xtol` (default 1e-8) and `gtol` (default 1e-5): the tolerances below which the run counts as converged.
- `maxiter` (default 100).

The cell below writes a renamed copy with `trust_radius = 3.0`, runs it and compares the objectives with a relative tolerance of 1e-6.

In [12]:
trust_text = config_trust.read_text()
big_text = trust_text.replace(
    'name = "rosenbrock_trust_region"', 'name = "rosenbrock_trust_region_radius3"'
)
big_text = big_text.replace(
    "options = { maxiter = 100 }", "options = { maxiter = 100, trust_radius = 3.0 }"
)
config_big = copied.folder / "rosenbrock_trust_region_radius3.toml"
config_big.write_text(big_text)
print("changed:", big_text != trust_text)

plan_big, summary_big = run_copy(config_big)
obj_1 = summary_trust.result["objective"]
obj_3 = summary_big.result["objective"]
print("objective with trust_radius 1.0:", obj_1)
print("objective with trust_radius 3.0:", obj_3)
if math.isclose(obj_1, obj_3, rel_tol=1e-6):
    print("equal to several digits: the radius was not what ended the first run")
else:
    print("different: the initial radius mattered")

changed: True
valid: True
trust_region on rosenbrock: 4 controls, max_evaluations = 502


job state: complete
objective with trust_radius 1.0: 3.701428613683974
objective with trust_radius 3.0: 3.7014286167153996
equal to several digits: the radius was not what ended the first run


If the two objectives are equal to several digits, the radius was not what ended the first run. The tolerances are the next suspects: write another renamed copy with a smaller `ftol` (for example `ftol = 1e-9`) in `options`, and, if that changes nothing, a smaller `gtol`. Change one option at a time, in a renamed copy.

## Step 8: when the landscape has many minima

Rosenbrock's difficulty is the slow valley to a single minimum. Other problems are multimodal. The `toy/rastrigin` example has a local minimum near every integer point. The population optimizers, the genetic algorithm (`ga`) and particle swarm (`pso`), search the whole range and can find the global minimum, where an ensemble gradient stops at the nearest local one. Its configurations are `rastrigin_ga.toml` and `rastrigin_pso.toml`.

First look at the two files, as in step 2.

In [13]:
rastrigin = copy_example(catalogue["toy/rastrigin"], Path("work"))
config_ga = rastrigin.folder / "rastrigin_ga.toml"
config_pso = rastrigin.folder / "rastrigin_pso.toml"

for label, path in (("genetic algorithm", config_ga), ("particle swarm", config_pso)):
    lines = [setting(line) for line in path.read_text().splitlines()]
    print(f"--- {label}: {rel(path)}")
    print("\n".join(line for line in lines if line))
    print()

--- genetic algorithm: work/toy/rastrigin/rastrigin_ga.toml
[optimization]
name = "rastrigin_ga"
seed = 1
[[controls.variables]]
name = "x"
initial = [3.0, -2.5]
lower = -5.12
upper = 5.12
[objective]
name = "rastrigin"
[optimizer]
name = "ga"
options = { maxiter = 200, transform = true }

--- particle swarm: work/toy/rastrigin/rastrigin_pso.toml
[optimization]
name = "rastrigin_pso"
seed = 1
[[controls.variables]]
name = "x"
initial = [3.0, -2.5]
lower = -5.12
upper = 5.12
[objective]
name = "rastrigin"
[optimizer]
name = "pso"
options = { maxiter = 200, transform = true }



The options of the population optimizers are about the population rather than a gradient:

- `pop_size` is the number of members of the population (default 20); the `initial` values of the controls are one of them. Each generation evaluates new members, so the cost grows with the population size and the number of generations.
- `maxiter` is the most generations to run (default 100). `stall` is the number of generations in a row without the best member improving by more than `ftol` (relative) after which the run counts as converged (default 20); `xtol` stops the run when the population has collapsed to a point.
- `init` is how the rest of the starting population is drawn within the bounds: `uniform` or `lhs` (Latin hypercube).
- `transform = true` makes the optimizer work in the unit cube of the bounds instead of the controls' own units: every control is rescaled to the range 0 to 1 between its `lower` and `upper`. This matters when controls have very different ranges, and it is switched off by default. Both Rastrigin files turn it on.
- `ga` also takes `n_offsprings` (children per generation, default `pop_size`) and the crossover and mutation settings (`crossover_prob`, `eta_c`, `mutation_prob`, `eta_m`).
- `pso` also takes the swarm weights `w`, `c1` and `c2`, the `topology` (`global` or `ring`) and `v_max`, the velocity limit as a fraction of each control's range.

The plan's `max_evaluations` is a ceiling built from these numbers, so a larger `pop_size` or `maxiter` raises it. Now run both with `run_copy`.

In [14]:
plan_ga, summary_ga = run_copy(config_ga)
plan_pso, summary_pso = run_copy(config_pso)

results = pd.DataFrame(
    [
        {
            "optimizer": p.optimizer,
            "max evaluations": p.max_evaluations,
            "objective": s.result["objective"],
            "controls": [round(float(v), 4) for v in s.result["x"]],
        }
        for p, s in ((plan_ga, summary_ga), (plan_pso, summary_pso))
    ]
)
print(results.to_string(index=False))

valid: True
ga on rastrigin: 2 controls, max_evaluations = 4020


job state: complete
valid: True
pso on rastrigin: 2 controls, max_evaluations = 4020


job state: complete
optimizer  max evaluations  objective         controls
       ga             4020   0.001114 [-0.0022, 0.001]
      pso             4020   0.000000      [-0.0, 0.0]


The global minimum of Rastrigin is 0 at the origin. An objective very close to 0 with controls close to `(0, 0)` means the global minimum was found. A run trapped in a local minimum would end near another integer point with an objective of about 1 or more. If both rows are near 0, a small difference between them is only the precision each reached, and one run each is too little to rank the two methods.

To try another optimizer on a problem, change only the configuration file you pass to `run_copy`.

## The same from the terminal

The commands below run after every cell above has run, from the folder that holds `work` and `apex_home`.

**Validate a configuration.** `apex validate` checks a configuration and plans its cost without simulating anything, as step 3 did. This one is the copy the notebook wrote in step 6 but never ran:

```bash
uv run apex validate work/optimization/rosenbrock_enopt_cli.toml
```

**Summarise runs.** `apex summary` lists every metric at every step of the runs it finds under the folders you give. For an optimization the steps are the optimization steps. Find the section of the run you care about, locate the objective among its metrics, and read its values from the first step to the last:

- A healthy run shows the objective falling quickly at first (for the Rosenbrock run, from about 282 at the start) and then levelling off.
- If it is still falling clearly at the last step, the run stopped early: raise `maxiter` in a renamed copy.
- If it is the same value from the first step on, or rises, change the step size (`alpha`) or the seed in a renamed copy, one at a time, as in step 6.
- A run that levels off at a value well above 0, as the Rosenbrock runs here do, has stopped along the valley rather than at the minimum: see the rule of thumb in step 5.
- The last step's objective should match the `objective` that `optimization_summary` gave you in the notebook for the same run.

This reads the runs the notebook made through the client:

```bash
uv run apex summary apex_home
```

**Report one run.** `apex report RUN_DIR` prints the metric history of a run and, with `--plots`, also writes figures (PNG and PDF) into `RUN_DIR/report` unless you pass `--output`. Here the run folder is found by listing `apex_home/optimizations/rosenbrock_enopt`, which holds one folder per run, named by its id:

```bash
uv run apex report "$(ls -d apex_home/optimizations/rosenbrock_enopt/* | head -n 1)" --plots
```

**Starting and continuing optimizations from the terminal.** The notebook does not run these, but you will meet them:

- `apex optimize CONFIG` starts an optimization directly from a configuration file. `--no-progress` turns off the progress display.
- `--max-iterations N` stops the run after N accepted iterations, that is, steps that passed backtracking and were taken. This differs from `maxiter`, which is the optimizer's own limit in the configuration. A run stopped this way is paused rather than converged, and can be continued.
- `apex optimize --resume RUN_DIR` continues the optimization run in `RUN_DIR`. Runs made through the client are in `apex_home/optimizations/<name>/<id>`.
- `apex run CONFIG` starts a history-matching run, not an optimization; `apex resume RUN_DIR` continues such a run. For an optimization, use `apex optimize --resume`.

To run a configuration a notebook already ran, copy it and give the copy another `name` in `[optimization]`, as in step 6.

## Pitfalls

- **Skipping validation.** A wrong name is caught at once by `validate_optimization`, but only after a whole run if you submit directly.
- **Judging by the states.** The job status and the run state describe the work, not the answer. Judge the objective against what you know of the problem.
- **Assuming more iterations always help.** If a run did not stop at its limit, raising the limit changes nothing, as step 6 checks.
- **Changing several options at once.** A worse result then does not say which change caused it; rerun with one change each, as in step 6.
- **Reading one run as proof.** Random methods give different results for different seeds; repeat with several seeds if it matters.
- **Comparing unequal budgets.** A plan can show `None` for the evaluations, and a planned maximum is not the number used. Then compare the objectives reached under stated stopping rules, and say so.
- **Rerunning under the same name.** Copy the configuration and change `name`.
- **Choosing an ensemble-gradient method for a problem with many local minima.** On Rastrigin-like problems it stops at the nearest minimum; use a global optimizer, as in step 8.
- **Looking in the wrong place.** Find results by job id through `client.optimizations()`, or list `apex_home/optimizations/<name>`, not by guessing folder names.
- **Running the terminal from another folder.** The relative paths `work` and `apex_home` depend on the folder you are in.

## What to read next

The examples catalogue has more optimization problems that follow the same workflow, including a family of multi-objective test problems (`toy/multi_objective/...`) solved by NSGA-II, NSGA-III and MOEA/D, and two toy-reservoir examples (`toy/toy_reservoir_npv_water` and `toy/toy_reservoir_robust_npv`) that trade NPV against water injected, or against its spread over model realizations. Copy one with `copy_example`, read its TOML as in step 2, and follow the same steps. The reference pages for the `enopt`, `trust_region`, `ga` and `pso` optimizers list every option with its default.